# Notebook 07 (Exp 2) — Comparison and Results

Aggregates Exp 1 vs Exp 2 results:
- Side-by-side metrics table (BLEU, ChrF, BERTScore F1)
- Bar chart: Exp 1 vs Exp 2 for LoRA and FFT
- Loss curve comparison
- Discussion of what changed and why

In [5]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT        = Path('..').resolve()
RESULTS_EXP1 = ROOT / 'outputs' / 'exp1' / 'results'
RESULTS_EXP2 = ROOT / 'outputs' / 'exp2' / 'results'
FIG_DIR      = RESULTS_EXP2 / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
print('Setup complete.')

Setup complete.


## 1. Load Scores (Exp 1 and Exp 2)

In [ ]:
exp1_path = RESULTS_EXP1 / 'bleu_scores.json'
if exp1_path.exists():
    with open(exp1_path) as f:
        exp1 = json.load(f)
else:
    print(f'⚠️  Exp1 scores not found at {exp1_path}')
    exp1 = None

with open(RESULTS_EXP2 / 'bleu_scores.json') as f:
    exp2 = json.load(f)

if exp1:
    print('Exp 1 LoRA:', exp1['lora'])
    print('Exp 1 FFT: ', exp1['fft'])
    print()
print('Exp 2 LoRA:', exp2['lora'])
print('Exp 2 FFT: ', exp2['fft'])

## 2. Comparison Table

In [ ]:
def get_row(scores, label):
    return {
        'Run':             label,
        'Corpus BLEU':     scores.get('corpus_bleu', '-'),
        'Corpus ChrF':     scores.get('corpus_chrf', '-'),
        'Sent BLEU Mean':  scores.get('sent_bleu_mean', '-'),
        'Sent BLEU P50':   scores.get('sent_bleu_p50', '-'),
        'BERTScore F1':    scores.get('bert_f1_mean', 'N/A (exp1 not scored)'),
        'BERTScore P':     scores.get('bert_p_mean', '-'),
        'BERTScore R':     scores.get('bert_r_mean', '-'),
    }

rows = []
if exp1:
    rows.extend([
        get_row(exp1['lora'], 'Exp1 — LoRA (3B, 3 epochs, LR=2e-4)'),
        get_row(exp1['fft'],  'Exp1 — FFT  (1.5B, 3 epochs, LR=2e-5)'),
    ])
rows.extend([
    get_row(exp2['lora'], 'Exp2 — LoRA (3B, 2 epochs+EarlyStopping, LR=2e-4)'),
    get_row(exp2['fft'],  'Exp2 — FFT  (1.5B, 3 epochs, LR=5e-5)'),
])
df = pd.DataFrame(rows).set_index('Run')

print('=== Full Comparison Table ===')
print(df.to_string())
df.to_csv(RESULTS_EXP2 / 'comparison_table.csv')
print('\nSaved to outputs/exp2/results/comparison_table.csv')

## 3. BLEU and ChrF Bar Chart (Exp 1 vs Exp 2)

In [ ]:
if exp1:
    labels    = ['LoRA\nExp1', 'LoRA\nExp2', 'FFT\nExp1', 'FFT\nExp2']
    bleu_vals = [exp1['lora']['corpus_bleu'], exp2['lora']['corpus_bleu'],
                 exp1['fft']['corpus_bleu'],  exp2['fft']['corpus_bleu']]
    chrf_vals = [exp1['lora']['corpus_chrf'], exp2['lora']['corpus_chrf'],
                 exp1['fft']['corpus_chrf'],  exp2['fft']['corpus_chrf']]
else:
    labels    = ['LoRA\nExp2', 'FFT\nExp2']
    bleu_vals = [exp2['lora']['corpus_bleu'], exp2['fft']['corpus_bleu']]
    chrf_vals = [exp2['lora']['corpus_chrf'], exp2['fft']['corpus_chrf']]

x     = np.arange(len(labels))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, vals, metric, color in zip(
    axes,
    [bleu_vals, chrf_vals],
    ['Corpus BLEU', 'Corpus ChrF'],
    ['steelblue', 'darkorange']
):
    bars = ax.bar(x, vals, width=0.5, color=color, edgecolor='white')
    for bar in bars:
        ax.annotate(f'{bar.get_height():.2f}',
                    xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylabel('Score')
    title = f'{metric} — Exp 1 vs Exp 2' if exp1 else f'{metric} — Exp 2'
    ax.set_title(title)
    ax.set_ylim(0, max(vals) * 1.35 if max(vals) > 0 else 1)

plt.tight_layout()
plt.savefig(FIG_DIR / 'exp2_bleu_chrf.png', dpi=150)
plt.show()

## 4. BERTScore Comparison (Exp 2 only — Exp 1 not scored)

In [ ]:
bert_keys   = ['bert_p_mean', 'bert_r_mean', 'bert_f1_mean']
bert_labels = ['Precision', 'Recall', 'F1']

lora_bert = [exp2['lora'].get(k, 0) for k in bert_keys]
fft_bert  = [exp2['fft'].get(k, 0)  for k in bert_keys]

x     = np.arange(len(bert_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - width/2, lora_bert, width, label='Exp2 LoRA', color='steelblue', edgecolor='white')
b2 = ax.bar(x + width/2, fft_bert,  width, label='Exp2 FFT',  color='darkorange', edgecolor='white')

for bar in [*b1, *b2]:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points',
                ha='center', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(bert_labels, fontsize=11)
ax.set_ylabel('BERTScore')
ax.set_title('BERTScore — Exp 2 LoRA vs FFT\n(distilbert-base-uncased, Modern\u2192Shakespeare)')
ax.legend(fontsize=11)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(FIG_DIR / 'exp2_bertscore.png', dpi=150)
plt.show()


## 5. Loss Curve Overlay (Exp 1 vs Exp 2)

In [ ]:
from PIL import Image
import os

EXP1_FIG_DIR = ROOT / 'outputs' / 'exp1' / 'results' / 'figures'

curves = {}
if exp1:
    curves.update({
        'Exp1 LoRA':  EXP1_FIG_DIR / 'lora_loss_curve.png',
        'Exp1 FFT':   EXP1_FIG_DIR / 'fft_loss_curve.png',
    })
curves.update({
    'Exp2 LoRA':  FIG_DIR / 'exp2_lora_loss_curve.png',
    'Exp2 FFT':   FIG_DIR / 'exp2_fft_loss_curve.png',
})

existing = {k: v for k, v in curves.items() if v.exists()}
if existing:
    n = len(existing)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, (title, path) in zip(axes, existing.items()):
        ax.imshow(Image.open(path))
        ax.axis('off')
        ax.set_title(title)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'loss_curves.png', dpi=150)
    plt.show()
else:
    print('Loss curve images not found — run training notebooks first.')

## 6. Discussion

| Dimension | Exp 1 | Exp 2 | Change |
|-----------|-------|-------|--------|
| **LoRA epochs** | 3 | 2 + EarlyStopping | Val loss rose from epoch 2 in exp1 — stopped earlier |
| **FFT LR** | 2e-5 | 5e-5 | Exp1 train≈val≈1.05 throughout; raised LR to break plateau |
| **Metric** | BLEU only | BLEU + BERTScore | BERTScore captures semantic similarity independent of surface form |
| **BLEU** | LoRA 0.09 / FFT 0.09 | Expected to improve slightly | Less overfitting / more learning |
| **BERTScore F1** | Not computed | Target: >0.80 | True quality signal for style transfer |

**Key insight**: BLEU is structurally unsuitable for style transfer (thousands of valid Shakespearean renderings per input). BERTScore is the primary quality metric for exp2.

**Further work**: (1) Multi-reference BLEU using paraphrase augmentation, (2) Unidirectional training (Mod→Shak only), (3) Larger r (32/64) for LoRA, (4) `roberta-large` BERTScore for publication.